In [2]:
# Import all packages

import bs4    # BeautifulSoup is used to scrape websites by parsing the HTML
import requests # Requests is used to make HTTP requests
import pandas as pd # Pandas is used to create "dataframes" which are used for data analysis

ModuleNotFoundError: No module named 'pandas'

In [8]:
# Making a request to the URL to get the entire HTML of the page

URL="https://www.hltv.org/stats/players" # URL of the page to be scraped
HEADER = {'User-Agent': 'Mozilla/5.0 (Windows; U; Windows NT 5.1; en-US; rv:1.9.0.7) Gecko/2009021910 Firefox/3.0.7'} # This is used to prevent the website from blocking the scraper
res = requests.get(URL, headers=HEADER) # This is the request to the URL with the headers passed in as a parameter
res.status_code # This will return the status code of the request (200 if okay)

In [12]:
# Parsing the HTML of the page

text=res.content # This is the HTML of the page
soup= bs4.BeautifulSoup(text) # This is the BeautifulSoup object that will be used to parse the HTML
soup.title.text, soup.h1.text # This will return the title and h1 of the page

In [16]:
# Checking total number of tables to be scraped

player_table= soup.find_all('table', attrs={"class": "stats-table player-ratings-table"}) # This is the table that contains the player data
len(player_table) # This will return the number of tables that were found
player_table=player_table[0] # This is the first and only table in the list of tables

In [30]:
# Dividing the tale

header= player_table.find("thead") # This is the header of the table
details= player_table.find("tbody") # This is the body of the table

In [32]:
# Checking out the header to see structure and number of columns in the

header 

<thead>
<tr class="stats-table-row">
<th class="playerCol">Player</th>
<th class="teamCol">Teams</th>
<th class="mapsCol">Maps</th>
<th class="rounds-col gtSmartphone-only">Rounds</th>
<th class="kdDiffCol">K-D Diff</th>
<th class="kdCol">K/D</th>
<th class="ratingCol">Rating<span class="ratingDesc">1.0</span></th>
</tr>
</thead>

In [40]:
# We will look for all table rows in details and then look for all table cells in each row

details= details.find_all("tr") # This is the list of table rows

In [161]:
# Define the columns of the dataframe

df = pd.DataFrame({}, columns=["Name", 
        "Teams Played In",
        "Number of Maps Played",
        "Number of Rounds Played",
        "Kill Death Difference",                                               ## CHANGE HERE
        "K/D Ratio",
        "HLTV Rating"])

df.columns


Index(['Name', 'Teams Played In', 'Number of Maps Played',
       'Number of Rounds Played', 'Kill Death Difference', 'K/D Ratio',
       'HLTV Rating'],
      dtype='object')

In [149]:
list=[]
for tr in details:
    #name
    name= tr.find_all('td', attrs={"class": "playerCol"})[0].find("a").text
    #teams
    teams= []
    team_td= tr.find_all('td', attrs={"class": "teamCol"})[0].find_all("a")
    for a in team_td:
        teams.append(a.find("img").get("title"))
    #maps played
    no_of_maps= tr.find_all('td', attrs={"class": "statsDetail"})[0].text
    #rounds played
    no_of_rounds= tr.find_all('td', attrs={"class": "gtSmartphone-only"})[0].text
    #kd difference
    kd_difference= tr.find_all('td', attrs={"class": "kdDiffCol"})[0].text
    #k/d
    kd= tr.find_all('td', attrs={"class": "statsDetail"})[2].text
    #HLTV Rating
    hltv_rating= tr.find_all('td', attrs={"class": "ratingCol"})[0].text
    #creating element
    element={
        "Name": name,
        "Teams Played In": teams,
        "Number of Maps Played": no_of_maps,
        "Number of Rounds Played": no_of_rounds,
        "Kill Death Difference": kd_difference,
        "K/D Ratio": kd,
        "HLTV Rating": hltv_rating
    }
    #print(element)
    #appending element
    list.append(element)

In [162]:
# Convert List of Dictionaries to DataFrame

df = df.append(list, ignore_index=True, sort=False) # This is the dataframe that will be used for data analysis
df

In [115]:
df['HLTV Rating']= df['HLTV Rating'].astype(str).astype(float) # This is to convert the HLTV Rating to a float

In [135]:
def prettier(list): # This is a function that will be used to format the dataframe
    return list[1:-1]

In [163]:
df['Teams Played In']=df['Teams Played In'].astype(str).apply(prettier) # This is to format the Teams Played In a list

In [176]:
df.to_csv("./CSGO Player Dataset.csv", index=False) # This is to save the dataframe as a CSV file